In [15]:
# ============================================================
# Jester_01_Load_and_Prepare.ipynb
# Part 1 — Imports + Folder Paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

# Root folder (where your Jester subfolders live)
JESTER_DIR = Path.home() / "Downloads" / "Joke_Project_Diss2026" / "Jester"

# Subfolders (from your screenshot)
RATINGS_DIR = JESTER_DIR / "jester_dataset_1_1"
JOKES_DIR   = JESTER_DIR / "jester_dataset_1_joke_texts"

print("JESTER_DIR:", JESTER_DIR, "exists?", JESTER_DIR.exists())
print("RATINGS_DIR:", RATINGS_DIR, "exists?", RATINGS_DIR.exists())
print("JOKES_DIR:", JOKES_DIR, "exists?", JOKES_DIR.exists())


JESTER_DIR: C:\Users\timil\Downloads\Joke_Project_Diss2026\Jester exists? True
RATINGS_DIR: C:\Users\timil\Downloads\Joke_Project_Diss2026\Jester\jester_dataset_1_1 exists? True
JOKES_DIR: C:\Users\timil\Downloads\Joke_Project_Diss2026\Jester\jester_dataset_1_joke_texts exists? True


In [16]:
# ============================================================
# Part 2 — Locate and Load the Ratings Matrix (.xls)
# ============================================================

# Find the ratings .xls file inside the ratings folder
xls_files = sorted(RATINGS_DIR.glob("*.xls"))

if not xls_files:
    raise FileNotFoundError(
        "No .xls file found in RATINGS_DIR. Check that you extracted the ratings zip."
    )

RATINGS_XLS = xls_files[0]
print("Using ratings file:", RATINGS_XLS)

# Load the ratings matrix
# Col 0 = number of jokes rated by user
# Col 1..100 = ratings for jokes 1..100
# Value 99 = not rated
ratings_df = pd.read_excel(RATINGS_XLS, header=None)

print("ratings_df shape:", ratings_df.shape)
ratings_df.head()


Using ratings file: C:\Users\timil\Downloads\Joke_Project_Diss2026\Jester\jester_dataset_1_1\jester-data-1.xls
ratings_df shape: (24983, 101)


,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,74,-7.82,8.79,-9.66,-8.16,-7.52,-8.50,-9.85,4.17,-8.98,...,2.82,99.00,99.00,99.00,99.00,99.00,-5.63,99.00,99.00,99.00
1,100,4.08,-0.29,6.36,4.37,-2.38,-9.66,-0.73,-5.34,8.88,...,2.82,-4.95,-0.29,7.86,-0.19,-2.14,3.06,0.34,-4.32,1.07
2,49,99.00,99.00,99.00,99.00,9.03,9.27,9.03,9.27,99.00,...,99.00,99.00,99.00,9.08,99.00,99.00,99.00,99.00,99.00,99.00
3,48,99.00,8.35,99.00,99.00,1.80,8.16,-2.82,6.21,99.00,...,99.00,99.00,99.00,0.53,99.00,99.00,99.00,99.00,99.00,99.00
4,91,8.50,4.61,-4.17,-5.39,1.36,1.60,7.04,4.61,-0.44,...,5.19,5.58,4.27,5.19,5.73,1.55,3.11,6.55,1.80,1.60


In [17]:
# ============================================================
# Part 3 — Convert Matrix → Edge List (user_id, joke_id, rating)
# ============================================================

# Convert to numpy for fast processing
R = ratings_df.to_numpy()

# Ratings only (skip the first "count rated" column)
ratings_only = R[:, 1:101]  # (num_users, 100)

# Keep entries that are real ratings (not NaN and not 99)
mask = (~np.isnan(ratings_only)) & (ratings_only != 99)

# Get coordinates of rated entries
user_idx, joke_col = np.where(mask)
vals = ratings_only[user_idx, joke_col]

# Build the long-format interactions table
edges = pd.DataFrame({
    "user_id": user_idx.astype(int),
    "joke_id": (joke_col + 1).astype(int),  # 1..100 to match init1..init100
    "rating": vals.astype(float)
})

print("Total rated edges:", len(edges))
edges.head()


Total rated edges: 1810455


,user_id,joke_id,rating
0,0,1,-7.82
1,0,2,8.79
2,0,3,-9.66
3,0,4,-8.16
4,0,5,-7.52


In [21]:
# ============================================================
# Part 4 — Load + Parse Joke Texts (robust + debug)
# ============================================================

import re

# 1) Find joke files but ignore macOS "._" metadata files
html_files = sorted(
    f for f in JOKES_DIR.rglob("*.html")
    if not f.name.startswith("._")
)

print("HTML files found (after filtering '._'):", len(html_files))
print("First 15 filenames:", [f.name for f in html_files[:15]])

if not html_files:
    raise FileNotFoundError(
        f"No usable .html files found under: {JOKES_DIR}\n"
        "Make sure the joke texts zip extracted correctly."
    )

def html_to_text(html: str) -> str:
    """Convert basic HTML into readable plain text."""
    html = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", html)
    html = re.sub(r"(?i)<br\s*/?>", "\n", html)
    html = re.sub(r"(?i)</p>", "\n", html)
    text = re.sub(r"(?s)<.*?>", " ", html)
    text = (text.replace("&nbsp;", " ")
                .replace("&quot;", '"')
                .replace("&amp;", "&")
                .replace("&lt;", "<")
                .replace("&gt;", ">"))
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+\n", "\n", text)
    return text.strip()

# 2) Parse files -> (joke_id, joke_text)
jokes = []
failed = []

for f in html_files:
    # Extract digits anywhere in the filename (works for init1, init001, init1 (1), etc.)
    m = re.search(r"(\d+)", f.stem)
    if not m:
        failed.append(f.name)
        continue

    joke_id = int(m.group(1))

    # Dataset 1 only uses joke IDs 1..100
    if not (1 <= joke_id <= 100):
        continue

    raw = f.read_text(encoding="utf-8", errors="ignore")
    jokes.append({"joke_id": joke_id, "source_file": f.name, "joke_text": html_to_text(raw)})

print("Parsed jokes:", len(jokes))
print("Failed ID extraction (first 10):", failed[:10])

if len(jokes) == 0:
    raise ValueError(
        "Parsed 0 jokes. The filenames shown above likely don't contain numbers.\n"
        "Paste the printed 'First 15 filenames' line and I'll adjust to your exact naming."
    )

# 3) Build dataframe + keep one file per joke_id
jokes_df = (
    pd.DataFrame(jokes)
      .sort_values(["joke_id", "source_file"])
      .drop_duplicates(subset=["joke_id"], keep="first")
      .sort_values("joke_id")
      .reset_index(drop=True)
)

print("jokes_df shape:", jokes_df.shape)

# 4) Quick sanity check (first joke)
if (jokes_df["joke_text"].str.contains("Mac OS X", na=False).any()):
    print("\nWARNING: Some parsed texts still look like mac metadata. We may still be reading wrong files.\n")

print("\nExample joke (id=1):")
print(jokes_df.loc[jokes_df["joke_id"] == 1, "joke_text"].iloc[0][:600], "...")


HTML files found (after filtering '._'): 100
First 15 filenames: ['init1.html', 'init10.html', 'init100.html', 'init11.html', 'init12.html', 'init13.html', 'init14.html', 'init15.html', 'init16.html', 'init17.html', 'init18.html', 'init19.html', 'init2.html', 'init20.html', 'init21.html']
Parsed jokes: 100
Failed ID extraction (first 10): []
jokes_df shape: (100, 3)

Example joke (id=1):
Joke 1 of 25
A man visits the doctor. The doctor says "I have bad news for you.You have
cancer and Alzheimer's disease".
The man replies "Well,thank God I don't have cancer!" ...


In [22]:
# ============================================================
# Part 5 — Join Ratings with Joke Text (sanity check)
# ============================================================

# Attach joke_text to every (user_id, joke_id, rating) row
edges_with_text = edges.merge(
    jokes_df[["joke_id", "joke_text"]],
    on="joke_id",
    how="left"
)

# Quick check: should be 0.0
missing_ratio = edges_with_text["joke_text"].isna().mean()
print("Missing joke text ratio:", missing_ratio)

# Print one example interaction
sample = edges_with_text.sample(1, random_state=42).iloc[0]
print("\n--- Random rated example ---")
print("User:", sample["user_id"])
print("Joke ID:", sample["joke_id"])
print("Rating:", sample["rating"])
print("\nJoke text:\n", sample["joke_text"][:600], "...")


Missing joke text ratio: 0.0

--- Random rated example ---
User: 323
Joke ID: 51
Rating: -6.55

Joke text:
 A Joke
Did you hear that Clinton has announced there is a new national bird?
 The spread eagle. ...


In [23]:
# ============================================================
# Part 6 — Save Clean Files (for next notebooks)
# ============================================================

# Save edges and jokes so we can reuse them in TF-IDF + LightGCN notebooks
out_edges = JESTER_DIR / "jester_edges_long.csv"
out_jokes = JESTER_DIR / "jester_jokes.csv"

edges.to_csv(out_edges, index=False)
jokes_df[["joke_id", "joke_text"]].to_csv(out_jokes, index=False)

print("Saved edges:", out_edges)
print("Saved jokes:", out_jokes)


Saved edges: C:\Users\timil\Downloads\Joke_Project_Diss2026\Jester\jester_edges_long.csv
Saved jokes: C:\Users\timil\Downloads\Joke_Project_Diss2026\Jester\jester_jokes.csv
